# Adaptive Retrieval

Classify the question first, then choose a retrieval strategy that fits it.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
STRATEGIES = {
    "factual":    "look up one specific fact",
    "analytical": "needs broad, multi-part coverage",
    "opinion":    "asks for recommendations or trade-offs",
    "no_retrieval": "general chat or maths that needs no documents",
}

def classify(query):
    options = "\n".join(f"- {k}: {v}" for k, v in STRATEGIES.items())
    label = llm(f"Classify the question into exactly one category. Reply with the category name only.\n{options}\n\nQuestion: {query}").lower()
    return next((k for k in STRATEGIES if k in label), "factual")

In [ ]:
def adaptive_rag(query):
    kind = classify(query)
    if kind == "no_retrieval":
        return kind, llm(query)
    if kind == "factual":
        contexts = [h["text"] for h in store.search(query, 2)]
    elif kind == "analytical":
        subs = llm_json(f"Split into 3 sub-questions. JSON list of strings.\n\n{query}") or [query]
        contexts = list({h["text"] for s in subs for h in store.search(s, 2)})
    else:  # opinion: gather diverse viewpoints with a wider net
        contexts = [h["text"] for h in store.search(query, 6)]
    return kind, answer(query, contexts)

for q in ["What dimension are all-MiniLM-L6-v2 vectors?", "Compare dense, BM25 and hybrid retrieval and how reranking fits in", "Should I fine-tune or use RAG?", "What is 12 times 12?"]:
    kind, response = adaptive_rag(q)
    print(f"[{kind}] {q}\n  {response[:300]}\n")